# E08 Modelo de referência da tecnologia

Medições dos enlaces da Fase 1 do projeto Copel. O SDK ajusta o modelo de RSSI a partir das configurações e das características dos enlaces. Este exemplo reutiliza features já extraídas, sem contato com os serviços. Requer SDK 0.1.46 com extras notebook. Abra este notebook na pasta notebooks do workspace.

In [ ]:
from cisei_planning_sdk import BaselineProject

project = BaselineProject("../projects/e08_baseline_copel")

## 1. Associar as medições às features existentes

O projeto contém features.json como cache de extração, no formato padrão cisei-feature-cache. O enriquecimento gera enriched_features.json, separado do cache original; treinamento e avaliação leem esse arquivo derivado. O arquivo data/e08/measurements.json fornece as configurações dos rádios e o RSSI observado. A receita baseline.json indica os dois caminhos.

A chamada abaixo valida a origem, o sentido, as coordenadas, as alturas, a frequência e as opções da extração antes de associar cada medição. A observação contém id único, nomes dos equipamentos, potência transmitida em dBm, ganhos em dBi e RSSI medido em dBm. O mesmo enlace pode ter várias observações. As features não são recalculadas nem alteradas. Repetir a chamada com os mesmos dados não duplica medições.

Na preparação manual deste exemplo foram mantidas 2.468 entradas de features e 2.278 medições com os campos de rádio obrigatórios. Outras 190 tinham ganhos de antena ausentes ou não numéricos; a lista está em data/e08/preparation_report.json. Nenhum ganho foi estimado para preencher essas lacunas. Os filtros do treinamento selecionam os mesmos 1.890 registros do exemplo anterior.

In [ ]:
project.enrich_features();

## 2. Ajustar o modelo

A receita baseline.json define os filtros e os ajustes de propagação, vegetação e edificações. O ajuste final utiliza os coeficientes anteriores como valores iniciais. O modelo é salvo em metrics/e08_copel_rssi.json. O gráfico apresenta os coeficientes e as quantidades de enlaces de cada etapa.

In [ ]:
project.train();
project.show_training();

## 3. Comparar previsão e medição

O limiar de -80 dBm é ilustrativo e deve ser definido pelo engenheiro. Selecione pontos no gráfico para consultar os enlaces e a contribuição dos componentes. Previsão boa com medição ruim orienta investigação; ambas ruins orientam revisão do projeto. Medições boas não indicam necessidade de correção. As estatísticas usam os dados do ajuste, sem separação de amostras de validação.

In [ ]:
project.evaluate(rssi_threshold_dbm=-80.0);
project.show_evaluation();

## 4. Disponibilizar a métrica para planejamento

O SDK grava metrics/e08_copel.toml, com a expressão do RSSI e sua conversão em custo positivo. O limiar e a escala de 10 dB por unidade de custo são parâmetros de engenharia deste exemplo. O JSON preserva o estimador; o TOML permite usar a previsão na métrica do planejador.

In [ ]:
project.export_metric(required_rssi_dbm=-80.0, db_per_cost=10.0);

## Uso posterior

A avaliação dos enlaces pode ser cruzada com os indicadores da aplicação. Quando há problemas de comunicação, auxilia a investigação corretiva. Quando a aplicação funciona satisfatoriamente, as indicações podem orientar manutenção preventiva ou melhorias de robustez. Os dados originais permanecem em data/e08; os resultados e as visualizações ficam na pasta do projeto.

## 5. Inspecionar os maiores desvios com o serviço de features

Esta seção é opcional e requer o serviço de features disponível. O painel reutiliza o visualizador de perfis, mapas e edificações. Os enlaces aparecem em ordem decrescente de diferença absoluta, com previsão e medição nos rótulos. A inspeção prepara somente o enlace selecionado; não recalcula o conjunto de treinamento.

A frequência de 900 MHz e a altura mínima de 7 m reproduzem os parâmetros usados na extração original. O filtro de 20 dB é ilustrativo. Use direction="below_expected" para investigar somente sinais inferiores à previsão, ou "above_expected" para o caso inverso. A diferença isolada não comprova defeito.

In [ ]:
from cisei_features_sdk.notebook import FeaturesNotebook

geo = FeaturesNotebook("http://features:8080", user=None)
inspector = project.inspect_divergent_links(
    geo, freq_mhz=900, minimum_height_m=7,
    min_abs_error_db=20, direction="both", limit=50,
);